In [1]:
import os
import sys

sys.path.append(os.getcwd())

In [2]:
from utils.data_loader import load_target_text
from taxonomy.taxonomy import Taxonomy
from models.tagrec import TagRec
from llm.llmJudge import LLMJudge
from ppi.ppi import LPPI
import pandas as pd

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
questions = load_target_text("data\\d1\\test_gold.csv", target_column = "eng", keep_columns=["taxonomy_gold"])
taxonomy = Taxonomy("taxonomy\\PCv3.txt")

In [4]:
# tagger = TagRec()
# tagger.set_taxonomy(taxonomy)
# tagger.load()

# tagrec_predictions = tagger.predict(questions, top_k=3)

# tagrec_predictions

In [5]:
llm_judge = LLMJudge(model="openai/gpt-oss-120b")

# test_judged = llm_judge.judge_taxonomy_predictions(tagrec_predictions)
# test_judged.to_pickle("main\\test_judged.pkl")

test_judged = pd.read_pickle("main\\test_judged.pkl")
test_judged

,target,taxonomy_gold,predicted_taxonomy,Subject_validity_llm,Chapter_validity_llm,Topic_validity_llm,Subject_validity_gold,Chapter_validity_gold,Topic_validity_gold
0,"Inspite of danger involved with hydrogen, it i...",TaxonomyNode(Chemistry >> Hydrogen >> Dihydrog...,TaxonomyNode(Chemistry >> Hydrogen >> Hydrogen...,0.95,0.92,0.15,1,1,0
1,"Inspite of danger involved with hydrogen, it i...",TaxonomyNode(Chemistry >> Hydrogen >> Dihydrog...,TaxonomyNode(Chemistry >> Hydrogen >> Properti...,0.95,0.88,0.44,1,1,0
2,"Inspite of danger involved with hydrogen, it i...",TaxonomyNode(Chemistry >> Hydrogen >> Dihydrog...,TaxonomyNode(Chemistry >> Hydrogen >> Dihydrogen),0.96,0.94,0.91,1,1,0
3,A circular coil of mean radius of \( 7 \mathrm...,TaxonomyNode(Physics >> Electromagnetic Induct...,TaxonomyNode(Physics >> Electromagnetic Induct...,0.99,0.96,0.94,1,1,0
4,A circular coil of mean radius of \( 7 \mathrm...,TaxonomyNode(Physics >> Electromagnetic Induct...,TaxonomyNode(Physics >> Electromagnetic Induct...,0.99,0.95,0.40,1,1,0
...,...,...,...,...,...,...,...,...,...
1498,If the ionization energy of hydrogen is\n\( 31...,TaxonomyNode(Chemistry >> Structure of Atom >>...,TaxonomyNode(Chemistry >> Structure of Atom >>...,0.95,0.90,0.45,1,1,0
1499,If the ionization energy of hydrogen is\n\( 31...,TaxonomyNode(Chemistry >> Structure of Atom >>...,TaxonomyNode(Chemistry >> Structure of Atom >>...,0.97,0.92,0.55,1,1,0
1500,Which curve corresponds to the\ntemperature de...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,0.99,0.98,0.97,1,1,0
1501,Which curve corresponds to the\ntemperature de...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,0.99,0.94,0.35,1,1,0


In [ ]:
lppi = LPPI(k=30, confidence=0.95)
lppi_gold = pd.read_pickle("data\\d1\\lppi_gold1.pkl")
lppi.fit(lppi_gold, score_cols=["Subject_validity", "Chapter_validity", "Topic_validity"])

Loading embedding model 'all-MiniLM-L6-v2' on cuda...
Fitting LPPI...
Embedding gold text...


Batches: 100%|██████████| 58/58 [00:03<00:00, 16.46it/s]


LPPI Fit complete.


In [ ]:
test_judged_rectified = lppi.calibrate(test_judged, clip_range=(0, 1))

Embedding unlabeled text...


Batches: 100%|██████████| 47/47 [00:02<00:00, 20.27it/s]


Finding nearest gold neighbors...
Applying local rectification...


In [8]:
test_judged_rectified

,target,taxonomy_gold,predicted_taxonomy,Subject_validity_llm,Chapter_validity_llm,Topic_validity_llm,Subject_validity_gold,Chapter_validity_gold,Topic_validity_gold,Subject_validity_rectified,Subject_validity_ci_size,Chapter_validity_rectified,Chapter_validity_ci_size,Topic_validity_rectified,Topic_validity_ci_size
0,"Inspite of danger involved with hydrogen, it i...",TaxonomyNode(Chemistry >> Hydrogen >> Dihydrog...,TaxonomyNode(Chemistry >> Hydrogen >> Hydrogen...,0.95,0.92,0.15,1,1,0,0.9178,0.048280,0.9132,0.058151,0.0090,0.087219
1,"Inspite of danger involved with hydrogen, it i...",TaxonomyNode(Chemistry >> Hydrogen >> Dihydrog...,TaxonomyNode(Chemistry >> Hydrogen >> Properti...,0.95,0.88,0.44,1,1,0,0.9178,0.048280,0.8732,0.058151,0.2990,0.087219
2,"Inspite of danger involved with hydrogen, it i...",TaxonomyNode(Chemistry >> Hydrogen >> Dihydrog...,TaxonomyNode(Chemistry >> Hydrogen >> Dihydrogen),0.96,0.94,0.91,1,1,0,0.9278,0.048280,0.9332,0.058151,0.7690,0.087219
3,A circular coil of mean radius of \( 7 \mathrm...,TaxonomyNode(Physics >> Electromagnetic Induct...,TaxonomyNode(Physics >> Electromagnetic Induct...,0.99,0.96,0.94,1,1,0,0.9760,0.028449,0.9470,0.036803,0.8002,0.081118
4,A circular coil of mean radius of \( 7 \mathrm...,TaxonomyNode(Physics >> Electromagnetic Induct...,TaxonomyNode(Physics >> Electromagnetic Induct...,0.99,0.95,0.40,1,1,0,0.9760,0.028449,0.9370,0.036803,0.2602,0.081118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1498,If the ionization energy of hydrogen is\n\( 31...,TaxonomyNode(Chemistry >> Structure of Atom >>...,TaxonomyNode(Chemistry >> Structure of Atom >>...,0.95,0.90,0.45,1,1,0,0.9636,0.065226,0.8942,0.042074,0.2590,0.094549
1499,If the ionization energy of hydrogen is\n\( 31...,TaxonomyNode(Chemistry >> Structure of Atom >>...,TaxonomyNode(Chemistry >> Structure of Atom >>...,0.97,0.92,0.55,1,1,0,0.9836,0.065226,0.9142,0.042074,0.3590,0.094549
1500,Which curve corresponds to the\ntemperature de...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,0.99,0.98,0.97,1,1,0,1.0038,0.041321,1.0170,0.074226,0.8624,0.092027
1501,Which curve corresponds to the\ntemperature de...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,TaxonomyNode(Chemistry >> Chemical Kinetics >>...,0.99,0.94,0.35,1,1,0,1.0038,0.041321,0.9770,0.074226,0.2424,0.092027


In [9]:
from utils.evaluator import RectificationEvaluator

evaluator = RectificationEvaluator(test_judged_rectified, score_columns=["Subject_validity", "Chapter_validity", "Topic_validity"])

In [10]:
evaluator.evaluate_all()

,MAE_llm,MAE_rectified,MAE_delta,RMSE_llm,RMSE_rectified,RMSE_delta,Spearman_llm,Spearman_rectified,Spearman_gain,Improvement_rate,Error_reduction_%,Directional_correctness,Confidence_weighted_error
Subject_validity,0.088217,0.087692,0.000525,0.237805,0.238028,-0.000224,0.353232,0.363916,0.010685,0.616766,0.005946,0.739854,2.785745
Chapter_validity,0.300246,0.303217,-0.002971,0.450005,0.463432,-0.013427,0.611808,0.617743,0.005934,0.481038,-0.009895,0.498337,7.156478
Topic_validity,0.344424,0.280972,0.063452,0.468511,0.394147,0.074364,0.393960,0.393327,-0.000633,0.779108,0.184227,0.916168,3.380943
